# AERO EYES — Pipeline Runner

**Hướng dẫn nhanh:**
1. `Runtime → Change runtime type → T4 GPU` (bắt buộc)
2. Chạy từng cell theo thứ tự từ trên xuống
3. Upload data của bạn vào Google Drive trước (xem hướng dẫn ở Section 2)

---
**Thông số nền tảng:**
| | Colab Free | Colab Pro | Kaggle Free |
|--|--|--|--|
| GPU | T4 (15 GB) | T4/A100 | T4 (16 GB) |
| RAM | 13 GB | 26 GB | 30 GB |
| Thời gian | ~12h/session | ~24h | 12h/week |
| Lưu file | Google Drive | Google Drive | Kaggle Dataset |

## Section 0 — Kiểm tra GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('CẢNH BÁO: Không tìm thấy GPU!')
    print('Vào Runtime → Change runtime type → chọn T4 GPU rồi chạy lại.')

## Section 1 — Clone Repo & Cài Dependencies

In [ ]:
# ============================================================
# THAY BẰNG URL REPO CỦA BẠN
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/aero_eyes_starter.git'
# ============================================================

import os
REPO_DIR = '/content/aero_eyes_starter'

if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    print('Repo đã tồn tại, pull latest...')
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# Cài PyTorch (Colab thường đã có, cell này chỉ upgrade nếu cần)
import torch
print(f'PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    print('Cài lại PyTorch với CUDA support...')
    !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q

In [ ]:
# Cài tất cả dependencies (~3-5 phút lần đầu)
!pip install -r requirements.txt -q
!pip install -e . -q

# opencv-contrib-python: cần cho OpenCV tracker (CSRT/KCF/MOSSE)
# Colab mặc định chỉ có opencv-python (không có tracker)
import cv2
_has_tracker = hasattr(cv2, 'legacy') and hasattr(cv2.legacy, 'TrackerCSRT_create')
if not _has_tracker:
    print('Cài opencv-contrib để có tracker...')
    !pip install opencv-contrib-python-headless -q
    print('Tracker OK — khởi động lại kernel nếu import lỗi')
else:
    print(f'OpenCV {cv2.__version__} — tracker OK')

print('Done!')

In [ ]:
# Kiểm tra import
import numpy, cv2, yaml
from pydantic import BaseModel
from aero_eyes.config import load_config
from aero_eyes.types import Box
print('Tất cả imports OK!')
print(f'  numpy {numpy.__version__}, cv2 {cv2.__version__}')

## Section 2 — Mount Google Drive & Chuẩn Bị Data

**Cấu trúc thư mục cần có trong Google Drive của bạn:**
```
MyDrive/
└── aero_eyes_data/
    ├── annotations (1).json      ← file ground truth
    ├── Backpack_0/
    │   ├── refs/
    │   │   ├── ref_0.jpg
    │   │   ├── ref_1.jpg
    │   │   └── ref_2.jpg
    │   └── video.mp4
    └── Person_1/
        ├── refs/ ...
        └── video.mp4
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted!')

In [ ]:
import os

# ============================================================
# CHỈNH ĐƯỜNG DẪN THEO CẤU TRÚC GOOGLE DRIVE CỦA BẠN
DRIVE_DATA_DIR = '/content/drive/MyDrive/aero_eyes_data'
ANNOTATIONS_FILE = f'{DRIVE_DATA_DIR}/annotations (1).json'
# ============================================================

# Tạo symlink để pipeline tìm thấy data
DATA_LINK = f'{REPO_DIR}/data'
if not os.path.exists(DATA_LINK):
    os.symlink(DRIVE_DATA_DIR, DATA_LINK)
    print(f'Symlink tạo: {DATA_LINK} → {DRIVE_DATA_DIR}')
else:
    print(f'Symlink đã tồn tại: {DATA_LINK}')

# Symlink cho annotations file
ANN_LINK = f'{REPO_DIR}/annotations (1).json'
if not os.path.exists(ANN_LINK):
    os.symlink(ANNOTATIONS_FILE, ANN_LINK)

# Kiểm tra
samples = [d for d in os.listdir(DATA_LINK)
           if os.path.isdir(f'{DATA_LINK}/{d}')]
print(f'Tìm thấy {len(samples)} sample(s): {samples[:5]}')

## Section 3 — Cấu Hình Pipeline

In [ ]:
# ============================================================
# CẤU HÌNH CHẠY — chỉnh theo nhu cầu
# ============================================================

SAMPLE_ID = None          # None = tất cả; hoặc 'Backpack_0'
ACCURACY_MODE = 'cheap_boosters'   # baseline | cheap_boosters | max_accuracy
PROPOSAL_MODEL = 'yolov11n'        # yolov11n | fastsam_s
DINOV2_VARIANT = 'vitb14'          # vitb14 (tốt) | vits14 (nhẹ)
BATCH_SIZE = 16

# ── SAHI ─────────────────────────────────────────────────────
USE_SAHI = True
SAHI_TILE_SIZE = 640    # 320 | 640 | 960
SAHI_OVERLAP = 0.25
# ─────────────────────────────────────────────────────────────

# ── Matching (Stage 3) ───────────────────────────────────────
# Adaptive threshold: tính ngưỡng theo phân phối similarity của từng video.
#   True  → ngưỡng = max(ADAPTIVE_MIN_FLOOR, mean + ADAPTIVE_Z * std)
#           Tự thích nghi với domain gap → ít FP hơn top-K thuần túy
#   False → dùng MATCH_THRESHOLD tuyệt đối
ADAPTIVE_THRESHOLD = True
ADAPTIVE_Z = 3.0          # càng cao → càng strict (ít FP, ít recall)
ADAPTIVE_MIN_FLOOR = 0.05 # ngưỡng tối thiểu tuyệt đối

# global_topk: sau khi lọc threshold, cap tối đa N candidates/video
# None = không cap  |  50 = lấy tối đa 50 tốt nhất
GLOBAL_TOPK = 50

# Chỉ dùng khi ADAPTIVE_THRESHOLD = False
MATCH_THRESHOLD = 0.55
# ─────────────────────────────────────────────────────────────

WORK_DIR = '/content/drive/MyDrive/aero_eyes_runs/exp001'
# ============================================================

_matching_desc = (
    f'adaptive z={ADAPTIVE_Z} floor={ADAPTIVE_MIN_FLOOR}'
    if ADAPTIVE_THRESHOLD else f'threshold={MATCH_THRESHOLD}'
)
print('Cấu hình:')
print(f'  Sample:    {SAMPLE_ID or "ALL"}')
print(f'  Accuracy:  {ACCURACY_MODE}')
print(f'  Proposal:  {PROPOSAL_MODEL}')
print(f'  DINOv2:    {DINOV2_VARIANT}')
print(f'  SAHI:      {"ON" if USE_SAHI else "OFF"}  (tile={SAHI_TILE_SIZE}px, overlap={SAHI_OVERLAP})')
print(f'  Matching:  {_matching_desc}  global_topk={GLOBAL_TOPK}')
print(f'  Work dir:  {WORK_DIR}')

In [ ]:
# Tạo overrides list từ config ở trên
overrides = [
    f'project.work_dir={WORK_DIR}',
    f'data.data_root={DATA_LINK}',
    f'accuracy.mode={ACCURACY_MODE}',
    f'stage1.feature_extractor.dinov2_variant={DINOV2_VARIANT}',
    f'stage2.proposal_model={PROPOSAL_MODEL}',
    f'runtime.batch_size={BATCH_SIZE}',
    f'stage2.sahi.use_sahi={str(USE_SAHI).lower()}',
    f'stage2.sahi.tile=[{SAHI_TILE_SIZE},{SAHI_TILE_SIZE}]',
    f'stage2.sahi.overlap={SAHI_OVERLAP}',
    f'stage3.adaptive_threshold={str(ADAPTIVE_THRESHOLD).lower()}',
    f'stage3.adaptive_z_score={ADAPTIVE_Z}',
    f'stage3.adaptive_min_floor={ADAPTIVE_MIN_FLOOR}',
    f'stage3.global_topk={GLOBAL_TOPK if GLOBAL_TOPK is not None else "null"}',
    f'stage3.match_threshold={MATCH_THRESHOLD}',
]
overrides_str = ' '.join(f'--set {o}' for o in overrides)
SAMPLE_STR = f'--sample {SAMPLE_ID}' if SAMPLE_ID else ''

print('CLI overrides:', overrides_str)

## Section 4 — Chạy Pipeline

In [ ]:
# Chạy TOÀN BỘ pipeline (Stage 1 → 5)
# Nếu bị ngắt giữa chừng: chạy lại cell này — use_cache=true sẽ bỏ qua
# các stage đã hoàn thành, tiếp tục từ stage bị dang dở.
!python -m aero_eyes.stages.run_all \
    --config configs/config.yaml \
    {SAMPLE_STR} \
    {overrides_str}

In [ ]:
# ── KHÔI PHỤC SAU KHI BỊ LỖI / NGẮT GIỮA CHỪNG ────────────────────────
# Dùng cell này khi pipeline bị lỗi ở một sample cụ thể.
# use_cache=true (mặc định) sẽ tự skip các stage đã có artifact.
# Chỉ cần chạy lại cell bên trên — không cần --from-stage.
#
# Nếu muốn bắt buộc chạy lại từ một stage cụ thể cho một sample:
RESUME_SAMPLE = 'Lifering_1'   # ← tên sample bị lỗi
RESUME_FROM   = 4              # ← stage muốn chạy lại từ (1-5)

!python -m aero_eyes.stages.run_all \
    --config configs/config.yaml \
    --sample {RESUME_SAMPLE} \
    --from-stage {RESUME_FROM} \
    {overrides_str}

In [ ]:
# --- HOẶC chạy từng stage riêng để dễ debug ---

# Thay SAMPLE_ID bằng tên sample muốn chạy
S = SAMPLE_ID or samples[0]  # dùng sample đầu tiên nếu không chỉ định
print(f'Chạy từng stage cho sample: {S}')

In [ ]:
# Stage 1 — Reference processing → prototype.npz
!python -m aero_eyes.stages.stage1 --config configs/config.yaml --sample {S} {overrides_str}

In [ ]:
# Stage 2 — Candidate generation → candidates.json  (lâu nhất)
!python -m aero_eyes.stages.stage2 --config configs/config.yaml --sample {S} {overrides_str}

In [ ]:
# Stage 3 — Cross-domain matching → detections.json
!python -m aero_eyes.stages.stage3 --config configs/config.yaml --sample {S} {overrides_str}

In [ ]:
# Stage 4 — Tracking → tracks.json
!python -m aero_eyes.stages.stage4 --config configs/config.yaml --sample {S} {overrides_str}

In [ ]:
# Stage 5 — Spatio-temporal output → submission.json
!python -m aero_eyes.stages.stage5 --config configs/config.yaml --sample {S} {overrides_str}

## Section 5 — Đánh Giá & Xem Kết Quả

In [ ]:
import os, json

# Liệt kê tất cả submission.json đã tạo
for root, dirs, files in os.walk(WORK_DIR):
    for f in files:
        if f == 'submission.json':
            path = os.path.join(root, f)
            with open(path) as fp:
                data = json.load(fp)
            n_frames = len(data[0]['annotations'][0]['bboxes']) if data else 0
            print(f'{path}  ({n_frames} frames with detection)')

In [ ]:
# Tính ST-IoU (cần có ground truth)
GT_FILE = ANN_LINK  # annotations (1).json

# Gom tất cả submission thành 1 file để eval toàn bộ
all_preds = []
for root, dirs, files in os.walk(WORK_DIR):
    for f in files:
        if f == 'submission.json':
            with open(os.path.join(root, f)) as fp:
                all_preds.extend(json.load(fp))

COMBINED_PRED = '/content/all_submissions.json'
with open(COMBINED_PRED, 'w') as fp:
    json.dump(all_preds, fp)
print(f'Gom {len(all_preds)} video vào {COMBINED_PRED}')

!python -m aero_eyes.evaluate \
    --pred {COMBINED_PRED} \
    --gt "{GT_FILE}" \
    --config configs/config.yaml

In [ ]:
# Tải file submission về máy local (nếu cần)
from google.colab import files
files.download(COMBINED_PRED)

## Section 6 — Chạy trên Dữ Liệu Giả (không cần dataset thật)

Dùng để test nhanh pipeline khi chưa có data thật.

In [ ]:
# Tạo synthetic fixture
!python -m scripts.make_synthetic_fixture --out tests/fixtures

# Chạy pipeline trên fixture
!python -m aero_eyes.stages.run_all \
    --config configs/config.yaml \
    --sample synth001 \
    --set data.data_root=tests/fixtures \
    --set project.work_dir=runs/synth_test

# Đánh giá
!python -m aero_eyes.evaluate \
    --pred runs/synth_test/synth001/submission.json \
    --gt tests/fixtures/synth001/gt.json \
    --config configs/config.yaml

# Chạy unit tests
!python -m pytest tests/test_st_iou.py -v